# DA-GPS ordinal225 training — Google Colab

Clean Colab runner for the **ordinal225** add-on experiment (hybrid tap loss, regulator territory attention bias, voltage violation MSE).

**Before you start:** `Runtime > Change runtime type > GPU` (T4 is fine). Mount Google Drive when prompted in the training cell.

## Ordinal225 coefficients
| Flag | Value |
|------|-------|
| `ATTN_REG_TERRITORY_BETA` | 7.0 |
| `REG_ORDINAL_ALPHA` | 2.25 |
| `VOLT_VIOLATION_ALPHA` | 6.0 |

## Logging
Sparse epoch logging: `--eval_every 10`, `--log_every 0` (validation metrics every 10 epochs; no per-batch spam).

## Run suffixes
- Smoke (`SMOKE_TEST=True`): `_addons_ordinal225_smoke`
- Full (`SMOKE_TEST=False`, default): `_addons_ordinal225_full`

Data caches and checkpoints use the same Colab Drive paths as `nonunique.ipynb` (`/content/drive/MyDrive/datasets_gnn2/...`).


In [ ]:
# Clone or update GNN-Sandia repo (run first)
import os

REPO_DIR = "/content/GNN-Sandia"
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"  # private? use a token URL

if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull origin main

%cd $REPO_DIR
os.environ["GNN2_REPO_ROOT"] = REPO_DIR
print(f"Working directory: {os.getcwd()}")
!git log --oneline -3


In [ ]:
import os
import sys
import subprocess
import datetime
import warnings
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_DATA = DRIVE_ROOT / "MyDrive/datasets_gnn2"
COLAB_CHUNK_DEFAULT = MYDRIVE_DATA / "original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
COLAB_NODE_PE_DEFAULT = (
    COLAB_CHUNK_DEFAULT
    / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
)
COLAB_RUNS_PARENT = Path("/content/GNN-Sandia/gnn2_architecture_search/attention checkpoints")
HOP_CSV_NAME = "load_hop_distance_to_each_regulator_all_index_nodes.csv"
WIN_CHUNK_DEFAULT = Path(r"D:\datasets\original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")


def _is_windows_drive_path(path) -> bool:
    s = str(path).strip().replace("/", "\\")
    return len(s) >= 2 and s[1] == ":" and s[0].isalpha()


def _on_colab() -> bool:
    return Path("/content").is_dir()


def _drive_mounted() -> bool:
    return DRIVE_ROOT.is_dir() and (DRIVE_ROOT / "MyDrive").is_dir()


def _resolve_data_path(path: Path, *, label: str, colab_fallback: Path | None = None) -> Path:
    """Resolve a data path; never join Windows drive letters with cwd on Linux."""
    raw = str(path)
    if _is_windows_drive_path(raw):
        if _on_colab() and colab_fallback is not None:
            warnings.warn(
                f"{label}={raw!r} is a Windows path on Linux/Colab; using {colab_fallback} instead.",
                UserWarning,
                stacklevel=2,
            )
            return colab_fallback.expanduser().resolve()
        if _on_colab():
            raise ValueError(
                f"{label}={raw!r} is a Windows absolute path and invalid on Colab. "
                f"Mount Drive and use e.g. {COLAB_CHUNK_DEFAULT}"
            )
        return Path(raw).expanduser().resolve()
    p = Path(path).expanduser()
    if p.is_absolute():
        return p.resolve()
    return (Path.cwd() / p).resolve()


def _find_node_pe_csv(chunk_parent: Path) -> Path:
    hits = sorted(chunk_parent.glob("run_*/gnn_node_index_master.csv"))
    if not hits:
        raise FileNotFoundError(
            f"No run_*/gnn_node_index_master.csv under {chunk_parent}"
        )
    return hits[0]


def _resolve_hop_csv(
    *,
    repo: Path,
    chunk_parent: Path,
    mydrive_data: Path | None,
) -> Path | None:
    """Resolve regulator hop CSV for --attn_reg_territory_bias (first existing candidate)."""
    env_raw = os.environ.get("GNN2_HOP_CSV", "").strip()
    candidates: list[Path] = []
    if env_raw:
        p = Path(env_raw).expanduser()
        candidates.append(p.resolve() if p.is_absolute() else (repo / p).resolve())
    if mydrive_data is not None:
        candidates.append((mydrive_data / HOP_CSV_NAME).resolve())
    cp = chunk_parent.resolve()
    candidates.extend(
        [
            (cp / ".." / HOP_CSV_NAME).resolve(),
            (cp.parent / HOP_CSV_NAME).resolve(),
        ]
    )
    candidates.extend(
        [
            (repo / "datasets_gnn2_from pc" / HOP_CSV_NAME).resolve(),
            (repo / "datasets_gnn2" / HOP_CSV_NAME).resolve(),
        ]
    )
    seen: set[Path] = set()
    for c in candidates:
        if c in seen:
            continue
        seen.add(c)
        if c.is_file():
            return c
    return None

def _sorted_run_chunk_dirs(chunk_parent: Path) -> list[Path]:
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")),
        key=lambda p: p.name,
    )


def _smoke_chunk_subdir_glob(chunk_parent: Path, count: int) -> str:
    """First `count` run_* names, comma-separated for --chunk_subdir_glob."""
    names = [p.name for p in _sorted_run_chunk_dirs(chunk_parent)]
    if len(names) < count:
        raise ValueError(
            f"SMOKE_CHUNK_COUNT={count} but only {len(names)} run_* under {chunk_parent}"
        )
    return ",".join(names[:count])


def _chunks_from_subdir_glob(chunk_parent: Path, glob_pat: str) -> list[Path]:
    import fnmatch

    glob_pat = str(glob_pat).strip()
    if "," in glob_pat:
        allowed = {s.strip() for s in glob_pat.split(",") if s.strip()}
        chunks = sorted(
            (p for p in chunk_parent.iterdir() if p.is_dir() and p.name in allowed),
            key=lambda p: p.name,
        )
        missing = allowed - {p.name for p in chunks}
        if missing:
            raise FileNotFoundError(f"Missing smoke chunk folders: {sorted(missing)}")
        return chunks
    return sorted(
        (p for p in chunk_parent.iterdir() if p.is_dir() and fnmatch.fnmatch(p.name, glob_pat)),
        key=lambda p: p.name,
    )


def _write_smoke_compare_manifest(
    runs_parent: Path,
    *,
    out_dir: Path,
    physics_weight: float,
    chunk_glob: str,
    n_chunks: int,
    epochs: int,
    seed: int,
) -> None:
    import json as _json

    path = runs_parent / "_smoke_compare_manifest.json"
    manifest = {}
    if path.is_file():
        manifest = _json.loads(path.read_text(encoding="utf-8"))
    key = "physics" if physics_weight > 0 else "baseline"
    manifest[key] = {
        "run_dir": str(out_dir.resolve()),
        "physics_weight": physics_weight,
        "chunk_glob": chunk_glob,
        "n_chunks": n_chunks,
        "epochs": epochs,
        "seed": seed,
    }
    path.write_text(_json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"Smoke compare manifest ({key}) -> {path}")


# Colab: mount Drive when chunks/caches live under MyDrive
if _on_colab() and not _drive_mounted():
    from google.colab import drive
    drive.mount("/content/drive")

# --- FULL training (set True only for quick smoke/debug) ---
SMOKE_TEST = False
SMOKE_CHUNK_COUNT = 3  # used only when SMOKE_TEST=True
SMOKE_EPOCHS = 15
SMOKE_PATIENCE = 5
SMOKE_SEED = 42
FULL_EPOCHS = 200
FULL_PATIENCE = 30
TRAIN_SEED = 42

# ORDINAL225 add-on experiment (2026-07): raise hybrid ordinal alpha from moderate 1.75 -> 2.25.
# Hypothesis: stronger ordinal tap-cost regularization improves reg tap accuracy without extreme settings.
ATTN_REG_TERRITORY_BETA = 7.0   # was 6.0 tuned / 9.0 extreme / 2.0 default
REG_ORDINAL_ALPHA = 2.25        # was 1.75 moderate / 1.25 tuned / 3.0 extreme / 1.0 default
VOLT_VIOLATION_ALPHA = 6.0      # was 8.0 tuned / 0.0 extreme (uniform MSE when 0)
_DA_CACHE_NAME = "da_gps_chunked_mvagg_smoke_gine" if SMOKE_TEST else "da_gps_chunked_mvagg_full_gine"

# --- paths: auto-detect Colab + Drive; override below if needed ---
_env_repo = os.environ.get("GNN2_REPO_ROOT", "").strip()
REPO = Path(_env_repo).expanduser().resolve() if _env_repo else Path.cwd().resolve()
if not (REPO / "train_da_gps_multitask_complex_voltage_gine.py").is_file():
    raise FileNotFoundError(f"Repo root missing trainer script: {REPO}")

if _on_colab():
    if not _drive_mounted():
        raise RuntimeError(
            "Colab requires Google Drive mounted. Re-run this cell and approve Drive access, "
            "or: from google.colab import drive; drive.mount('/content/drive')"
        )
    CHUNK_PARENT = COLAB_CHUNK_DEFAULT
    NODE_PE_CSV = COLAB_NODE_PE_DEFAULT
    # Caches must live on Drive to reuse across Colab sessions (/content is wiped on reset)
    DA_CACHE_ROOT = MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = COLAB_RUNS_PARENT
elif os.name == "nt":
    CHUNK_PARENT = WIN_CHUNK_DEFAULT
    NODE_PE_CSV = None  # auto: first run_*/gnn_node_index_master.csv under CHUNK_PARENT
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"
else:
    CHUNK_PARENT = REPO / "datasets_gnn2_from pc/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40"
    NODE_PE_CSV = None
    DA_CACHE_ROOT = REPO / f"datasets_gnn2_from pc/cache/{_DA_CACHE_NAME}"
    GNN_CACHE_ROOT = REPO / "datasets_gnn2_from pc/cache/gnn_only_chunked_mvagg_full_gine"
    RUNS_PARENT = REPO / "gnn2_architecture_search/attention checkpoints"

# Optional overrides (use POSIX paths on Colab, not D:\...):
# CHUNK_PARENT = Path("/content/drive/MyDrive/datasets_gnn2/original_8500_unbalanced_chunked_no_bess_new_diverse_2000_40")
# NODE_PE_CSV = CHUNK_PARENT / "run_001_scen_0000_0049_seed_20420233/gnn_node_index_master.csv"
# RUNS_PARENT = MYDRIVE_DATA / "runs"  # persist checkpoints on Drive instead of ephemeral clone

# --- lighter architecture (locked in for add-ons full run) ---
LIGHTER_HIDDEN = 64
LIGHTER_LAYERS = 2
LIGHTER_HEADS = 2
LIGHTER_NODE_EMB_DIM = 2
LIGHTER_EDGE_EMB_DIM = 0

PHYSICS_WEIGHT = 0.0  # baseline; early_stop_on=total

# Explicit PF balance nodes when physics on (1177 hetero MV load nodes; chunk-safe)
PF_BALANCE_NODE_LIST_CSV = REPO / "colab_pf_data/pf_balance_nodes_explicit.csv"

META_AUX_COLS = "pv_pv2_p_post_kw,pv_pv2_q_post_kvar,P_loss_total_post_kw,Q_loss_total_post_kvar"
NUM_WORKERS = 0 if os.name == "nt" else 4

os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ.setdefault("GNN2_REPO_ROOT", str(REPO))

_colab_fb = COLAB_CHUNK_DEFAULT if _on_colab() else None
chunk_parent = _resolve_data_path(CHUNK_PARENT, label="CHUNK_PARENT", colab_fallback=_colab_fb)
if SMOKE_TEST:
    CHUNK_GLOB = _smoke_chunk_subdir_glob(chunk_parent, SMOKE_CHUNK_COUNT)
    EPOCHS = SMOKE_EPOCHS
    PATIENCE = SMOKE_PATIENCE
    SEED = SMOKE_SEED
else:
    CHUNK_GLOB = "run_*"
    EPOCHS = FULL_EPOCHS
    PATIENCE = FULL_PATIENCE
    SEED = TRAIN_SEED
da_cache_root = _resolve_data_path(
    DA_CACHE_ROOT,
    label="DA_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / f"cache/{_DA_CACHE_NAME}" if _on_colab() else None,
)
gnn_cache_root = _resolve_data_path(
    GNN_CACHE_ROOT,
    label="GNN_CACHE_ROOT",
    colab_fallback=MYDRIVE_DATA / "cache/gnn_only_chunked_mvagg_full_gine" if _on_colab() else None,
)
runs_parent = _resolve_data_path(
    RUNS_PARENT,
    label="RUNS_PARENT",
    colab_fallback=MYDRIVE_DATA / "runs" if _on_colab() else None,
)

# --- PF topology root (repo colab_pf_data/ after git pull, or Drive dailyagg fallback) ---
from gnn2_pf_data_paths import PF_CAP_NODES_REL, PF_REG_CATALOG_REL, resolve_pf_catalog_paths

PF_DATA_ROOT = None
if PHYSICS_WEIGHT > 0:
    _reg_cat, _cap_map, PF_DATA_ROOT = resolve_pf_catalog_paths(
        repo=REPO,
        preferred_root=None,
        chunk_parent=chunk_parent,
    )

print("=== Preflight (add-ons ordinal225 experiment) ===")
print(f"ADDON COEFFS:   territory_beta={ATTN_REG_TERRITORY_BETA} ordinal_alpha={REG_ORDINAL_ALPHA} volt_violation_alpha={VOLT_VIOLATION_ALPHA}")
print(f"REPO:           {REPO}")
print(f"SMOKE_TEST:     {SMOKE_TEST}")
print(f"SMOKE_CHUNK_COUNT: {SMOKE_CHUNK_COUNT}")
print(f"CHUNK_GLOB:     {CHUNK_GLOB}")
print(f"EPOCHS:         {EPOCHS}")
print(f"PATIENCE:       {PATIENCE}")
print(f"SEED:           {SEED}")
print(f"CHUNK_PARENT:   {chunk_parent}")
if not chunk_parent.is_dir():
    raise FileNotFoundError(f"CHUNK_PARENT not found: {chunk_parent}")

run_preview = sorted(
    p.name for p in chunk_parent.iterdir() if p.is_dir() and p.name.startswith("run_")
)[:5]
print(f"run_* preview:  {run_preview}")

if NODE_PE_CSV is None:
    node_pe = _find_node_pe_csv(chunk_parent)
else:
    node_pe = _resolve_data_path(NODE_PE_CSV, label="NODE_PE_CSV", colab_fallback=COLAB_NODE_PE_DEFAULT if _on_colab() else None)
    if not node_pe.is_file():
        warnings.warn(
            f"NODE_PE_CSV not found at {node_pe}; auto-discovering under CHUNK_PARENT.",
            UserWarning,
            stacklevel=2,
        )
        node_pe = _find_node_pe_csv(chunk_parent)

print(f"NODE_PE_CSV:    {node_pe}")
print(f"DA_CACHE_ROOT:  {da_cache_root}")
print(f"GNN_CACHE_ROOT: {gnn_cache_root}")
print(f"RUNS_PARENT:    {runs_parent}")

_mydrive_for_hop = MYDRIVE_DATA if _on_colab() and _drive_mounted() else None
hop_csv = _resolve_hop_csv(repo=REPO, chunk_parent=chunk_parent, mydrive_data=_mydrive_for_hop)
_hop_status = "OK" if hop_csv is not None else "MISSING"
print(f"HOP_CSV:        {_hop_status}" + (f"  {hop_csv}" if hop_csv else ""))
if hop_csv is None:
    _expected = MYDRIVE_DATA / HOP_CSV_NAME if _on_colab() else REPO / "datasets_gnn2_from pc" / HOP_CSV_NAME
    raise FileNotFoundError(
        "Regulator hop CSV required for --attn_reg_territory_bias but not found. "
        f"Generate with compute_hop_distance_all_index_nodes.py or copy to {_expected}"
    )
os.environ["GNN2_HOP_CSV"] = str(hop_csv)

from da_gps_hop_attention_ratios import TARGET_REG_COLS, validate_reg_hop_csv

_reg_csv = None
for _cand in (
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
):
    if _cand.is_file():
        _reg_csv = _cand
        break
_hop_map = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    regulator_csv=_reg_csv,
)
print("HOP preflight (phase-consistent):", ", ".join(f"{k}->{v}" for k, v in _hop_map.items()))

# Phase-consistent hop column mapping preflight (territory bias is harmful if wrong).
import pandas as pd
from da_gps_hop_attention_ratios import REG_COL_TO_HOP_COL, validate_reg_hop_csv
from train_da_gps_multitask_complex_voltage import TARGET_REG_COLS

_hop_node_names = pd.read_csv(node_pe)["node"].astype(str).tolist()
_reg_csv_candidates = [
    chunk_parent.parent / "regulator_involved_nodes.csv",
    REPO / "datasets_gnn2_from pc" / "loadtype_8500_dailyagg" / "regulator_involved_nodes.csv",
]
_hop_reg_csv = next((p for p in _reg_csv_candidates if p.is_file()), None)
_hop_pairs = validate_reg_hop_csv(
    hop_csv,
    list(TARGET_REG_COLS),
    node_names=_hop_node_names,
    reg_col_to_hop_col=REG_COL_TO_HOP_COL,
    regulator_csv=_hop_reg_csv,
)
print("HOP mapping preflight (reg_col -> hop_col):")
for _rc, _hc in _hop_pairs.items():
    print(f"  {_rc} -> {_hc}")

if PHYSICS_WEIGHT > 0:
    assert PF_DATA_ROOT is not None
    _pf_checks = [
        ("reg_catalog", PF_DATA_ROOT / PF_REG_CATALOG_REL),
        ("cap_nodes", PF_DATA_ROOT / PF_CAP_NODES_REL),
        ("electrical_distance", PF_DATA_ROOT / "electrical_distance_from_substation.csv"),
        (
            "hetero_mv_nodes",
            PF_DATA_ROOT / "Heterogenous GNN dataset/nodes/hetero_mv_nodes_load_transformer.csv",
        ),
        ("bus_kv_cache", PF_DATA_ROOT / "bus_kv_base_by_node.csv"),
    ]
    print(f"PF_DATA_ROOT:   {PF_DATA_ROOT}")
    for label, p in _pf_checks:
        status = "OK" if p.is_file() else "MISSING"
        print(f"  PF {label}: {status}  {p}")
        if not p.is_file():
            raise FileNotFoundError(f"Physics preflight missing {label}: {p}")
else:
    print("PF_DATA_ROOT:   (skipped — PHYSICS_WEIGHT=0)")

print("=================")

da_cache_root.mkdir(parents=True, exist_ok=True)
gnn_cache_root.mkdir(parents=True, exist_ok=True)
runs_parent.mkdir(parents=True, exist_ok=True)

chunks = _chunks_from_subdir_glob(chunk_parent, CHUNK_GLOB)
if not chunks:
    raise RuntimeError(f"No folders for CHUNK_GLOB={CHUNK_GLOB!r} under {chunk_parent}")

for p in chunks:
    for name in (
        "gnn_node_features_and_targets_mvagg.csv",
        "gnn_edges_phase_static.csv",
        "gnn_sample_meta.csv",
    ):
        if not (p / name).is_file():
            raise FileNotFoundError(f"Missing {name} in {p.name}")

print(f"Found {len(chunks)} chunk(s) for CHUNK_GLOB under {chunk_parent}")

tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_pf_suffix = "_pf" if PHYSICS_WEIGHT > 0 else ""
_addon_run_suffix = "_addons_ordinal225_smoke" if SMOKE_TEST else "_addons_ordinal225_full"
out_dir = runs_parent / (
    f"da_gps_chunked_l{LIGHTER_LAYERS}_h{LIGHTER_HIDDEN}_mvagg_gine_metaaux_regce{_pf_suffix}{_addon_run_suffix}_{tag}"
)
out_dir.mkdir(parents=True, exist_ok=True)

_early_stop = "voltage" if PHYSICS_WEIGHT > 0 else "total"

cmd = [
    sys.executable, "-u", "train_da_gps_multitask_complex_voltage_gine.py",
    "--chunk_parent", str(chunk_parent),
    "--chunk_subdir_glob", CHUNK_GLOB,
    "--nodes_csv", "gnn_node_features_and_targets_mvagg.csv",
    "--edge_catalog_csv", "gnn_edges_phase_static.csv",
    "--meta_csv", "gnn_sample_meta.csv",
    "--node_feature_cols", "p_load_kw,q_load_kvar,p_pv_kw",
    "--exclude_bess_features",
    "--node_pe_csv", str(node_pe),
    "--node_pe_cols", "auto",
    "--n_system_tokens", "10",
    "--aux_meta_cols", META_AUX_COLS,
    "--lambda_pv", "0.1",
    "--out_dir", str(out_dir),
    "--cache_dir", str(da_cache_root),
    "--bootstrap_gnn_cache_dir", str(gnn_cache_root),
    "--epochs", str(EPOCHS),
    "--batch_size", "64",
    "--hidden", str(LIGHTER_HIDDEN),
    "--layers", str(LIGHTER_LAYERS),
    "--heads", str(LIGHTER_HEADS),
    "--node_emb_dim", str(LIGHTER_NODE_EMB_DIM),
    "--edge_emb_dim", str(LIGHTER_EDGE_EMB_DIM),
    "--lr", "5e-4",
    "--weight_decay", "1e-5",
    "--lambda_cap", "0.1",
    "--lambda_reg", "0.1",
    "--reg_loss", "ce",
    "--per_device_cap_head",
    "--per_device_reg_head",
    "--patience", str(PATIENCE),
    "--seed", str(SEED),
    "--train_frac", "0.80",
    "--val_frac", "0.10",
    "--sample_frac", "1.0",
    "--num_workers", str(NUM_WORKERS),
    "--eval_every", "10",
    "--log_every", "0",
    "--checkpoint_every", "10",
    "--early_stop_on", _early_stop,
    "--dropout", "0.1",
    "--hop_csv", str(hop_csv),
    "--attn_reg_territory_bias",
    "--attn_reg_territory_beta", str(ATTN_REG_TERRITORY_BETA),
    "--reg_hybrid_tap_loss",
    "--reg_ordinal_alpha", str(REG_ORDINAL_ALPHA),
    "--volt_violation_weight",
    "--volt_lo_pu", "0.96",
    "--volt_hi_pu", "1.04",
    "--volt_violation_alpha", str(VOLT_VIOLATION_ALPHA),
]

if PHYSICS_WEIGHT > 0:
    _pf_flags = [
        "--pf_data_root", str(PF_DATA_ROOT),
        "--loss_power_balance_weight", str(PHYSICS_WEIGHT),
        "--pf_sparse_y", "1",
        "--pf_huber_delta_kw", "10",
        "--pf_detach_controls",
    ]
    _bal = Path(PF_BALANCE_NODE_LIST_CSV)
    if not _bal.is_absolute():
        _bal = (REPO / _bal).resolve()
    if not _bal.is_file():
        raise FileNotFoundError(f"PF_BALANCE_NODE_LIST_CSV not found: {_bal}")
    _pf_flags.extend(["--pf_balance_node_list_csv", str(_bal)])
    cmd.extend(_pf_flags)

_mode = "physics-informed" if PHYSICS_WEIGHT > 0 else "baseline"
print(f"PHYSICS_WEIGHT={PHYSICS_WEIGHT} ({_mode})")
print(f"early_stop_on={_early_stop}")
print("\nRunning:\n ", " ".join(cmd), "\n", flush=True)

with subprocess.Popen(
    cmd,
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
) as proc:
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()

if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

print("\nTraining completed.")
print("Run dir:", out_dir.resolve())
print("Checkpoint (best):", (out_dir / "da_gps_multitask_best.pt").resolve())
print("Checkpoint (last):", (out_dir / "training_last.pt").resolve())
print("Regulator classes:", (out_dir / "reg_class_tables.json").resolve())
print("Report:", (out_dir / "da_gps_report.json").resolve())
_run_name = out_dir.name
print("\n=== Section 8 prerequisite ===")
print(f"Baseline run folder:  {out_dir.resolve()}")
print(f"Section 8 expects:    {MYDRIVE_DATA / 'runs' / _run_name}")
print("Checkpoint for init:  da_gps_multitask_best.pt (or training_last.pt)")
if _on_colab() and not str(out_dir.resolve()).startswith(str(MYDRIVE_DATA.resolve())):
    print(
        "WARNING: run dir is NOT under MyDrive/datasets_gnn2 — "
        "copy to Drive before disconnecting the VM or Section 8 will fail."
    )
else:
    print("Checkpoints are on Drive; safe to run Section 8 after this session ends.")
_write_smoke_compare_manifest(
    runs_parent,
    out_dir=out_dir,
    physics_weight=PHYSICS_WEIGHT,
    chunk_glob=CHUNK_GLOB,
    n_chunks=len(chunks),
    epochs=EPOCHS,
    seed=SEED,
)
